In [ ]:
import pandas as pd

df = pd.read_csv(r"D:\DEPI\mini project\telco.csv")  
print("Shape before dropping columns:", df.shape)
columns_to_drop = [
    'Customer ID', 
    'Latitude', 
    'Longitude', 
    'Zip Code', 
    'Country', 
    'State'
]

df_cleaned = df.drop(columns=columns_to_drop, errors='ignore')

print("Shape after dropping columns:", df_cleaned.shape)

print(df_cleaned.columns.tolist())

Shape before dropping columns: (7043, 50)
Shape after dropping columns: (7043, 44)
['Gender', 'Age', 'Under 30', 'Senior Citizen', 'Married', 'Dependents', 'Number of Dependents', 'City', 'Population', 'Quarter', 'Referred a Friend', 'Number of Referrals', 'Tenure in Months', 'Offer', 'Phone Service', 'Avg Monthly Long Distance Charges', 'Multiple Lines', 'Internet Service', 'Internet Type', 'Avg Monthly GB Download', 'Online Security', 'Online Backup', 'Device Protection Plan', 'Premium Tech Support', 'Streaming TV', 'Streaming Movies', 'Streaming Music', 'Unlimited Data', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charge', 'Total Charges', 'Total Refunds', 'Total Extra Data Charges', 'Total Long Distance Charges', 'Total Revenue', 'Satisfaction Score', 'Customer Status', 'Churn Label', 'Churn Score', 'CLTV', 'Churn Category', 'Churn Reason']


In [16]:
import pandas as pd
import numpy as np

missing_values = df_cleaned.isnull().sum()
missing_percentage = (missing_values / len(df_cleaned)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_values,
    'Percentage (%)': missing_percentage
})

print(missing_df[missing_df['Missing Count'] > 0].sort_values(by='Missing Count', ascending=False))

numerical_cols = df_cleaned.select_dtypes(include=['int64', 'float64']).columns
for col in numerical_cols:
    if df_cleaned[col].isnull().sum() > 0:
        df_cleaned[col].fillna(df_cleaned[col].median(), inplace=True)

categorical_cols = df_cleaned.select_dtypes(include=['object', 'category']).columns
for col in categorical_cols:
    if df_cleaned[col].isnull().sum() > 0:
        df_cleaned[col].fillna(df_cleaned[col].mode()[0], inplace=True)

print(df_cleaned.isnull().sum().sum())

                Missing Count  Percentage (%)
Churn Category           5174       73.463013
Churn Reason             5174       73.463013
Offer                    3877       55.047565
Internet Type            1526       21.666903
0


C:\Users\hp\AppData\Local\Temp\ipykernel_27244\1052023962.py:22: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_cleaned[col].fillna(df_cleaned[col].mode()[0], inplace=True)


In [ ]:

y = df_cleaned['Churn Label'].apply(lambda x: 1 if str(x).strip().lower() == 'yes' else 0)

columns_to_remove = [
    'Customer',
    'Customer ID',
    'Churn Value', 
    'Churn Label', 
    'Churn Score', 
    'Churn Reason', 
    'Customer Status', 
    'Satisfaction Score', 
    'Churn Category'
]

X = df_cleaned.drop(columns=columns_to_remove, errors='ignore')

print("عدد الـ Features بعد التصفية:", X.shape[1])
print("شكل الـ Target (y):", y.shape)
print("\nقائمة الـ Features الفعلية اللي هندخلها للموديل:")
print(X.columns.tolist())

عدد الـ Features بعد التصفية: 38
شكل الـ Target (y): (7043,)

قائمة الـ Features الفعلية اللي هندخلها للموديل:
['Gender', 'Age', 'Under 30', 'Senior Citizen', 'Married', 'Dependents', 'Number of Dependents', 'City', 'Population', 'Quarter', 'Referred a Friend', 'Number of Referrals', 'Tenure in Months', 'Offer', 'Phone Service', 'Avg Monthly Long Distance Charges', 'Multiple Lines', 'Internet Service', 'Internet Type', 'Avg Monthly GB Download', 'Online Security', 'Online Backup', 'Device Protection Plan', 'Premium Tech Support', 'Streaming TV', 'Streaming Movies', 'Streaming Music', 'Unlimited Data', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charge', 'Total Charges', 'Total Refunds', 'Total Extra Data Charges', 'Total Long Distance Charges', 'Total Revenue', 'CLTV']


In [ ]:
import pandas as pd

X_encoded = X.copy()

binary_columns = []
for col in X_encoded.select_dtypes(include=['object']).columns:
    unique_vals = X_encoded[col].dropna().unique()
    if len(unique_vals) == 2:
        binary_columns.append(col)

print( binary_columns)

for col in binary_columns:
    vals = X_encoded[col].unique()
    positive_val = 'Yes' if 'Yes' in vals else vals[0]
    X_encoded[col] = X_encoded[col].apply(lambda x: 1 if x == positive_val else (0 if pd.notnull(x) else x))

categorical_cols_to_encode = [col for col in X_encoded.select_dtypes(include=['object', 'category']).columns]

print( categorical_cols_to_encode)

# استخدام دالة get_dummies للـ One-Hot Encoding
X_encoded = pd.get_dummies(X_encoded, columns=categorical_cols_to_encode, drop_first=True)

print( X_encoded.shape)

['Gender', 'Under 30', 'Senior Citizen', 'Married', 'Dependents', 'Referred a Friend', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection Plan', 'Premium Tech Support', 'Streaming TV', 'Streaming Movies', 'Streaming Music', 'Unlimited Data', 'Paperless Billing']
['City', 'Quarter', 'Offer', 'Internet Type', 'Contract', 'Payment Method']
(7043, 1147)


In [19]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

print("شكل X_train:", X_train.shape)
print("شكل X_test:", X_test.shape)
StandardScaler
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nتم تقسيم الداتا وعمل Scaling بنجاح وجاهزين لبناء أول Machine Learning Model!")

شكل X_train: (5634, 1147)
شكل X_test: (1409, 1147)

تم تقسيم الداتا وعمل Scaling بنجاح وجاهزين لبناء أول Machine Learning Model!


In [20]:
print("Original target distribution:")
print(y.value_counts())

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Original target distribution:
Churn Label
0    5174
1    1869
Name: count, dtype: int64

Training target distribution:
Churn Label
0    4139
1    1495
Name: count, dtype: int64

Testing target distribution:
Churn Label
0    1035
1     374
Name: count, dtype: int64


In [21]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)


accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%\n")

print("Classification Report:")
print(classification_report(y_test, y_pred))

Model Accuracy: 83.18%

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.94      0.89      1035
           1       0.77      0.52      0.62       374

    accuracy                           0.83      1409
   macro avg       0.81      0.73      0.76      1409
weighted avg       0.83      0.83      0.82      1409



In [22]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

knn_model = KNeighborsClassifier(n_neighbors=5)


knn_model.fit(X_train_scaled, y_train)

y_pred_knn = knn_model.predict(X_test_scaled)


accuracy_knn = accuracy_score(y_test, y_pred_knn)
print(f"KNN Model Accuracy: {accuracy_knn * 100:.2f}%\n")

print("Classification Report (KNN):")
print(classification_report(y_test, y_pred_knn))

C:\Users\hp\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\hp\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\hp\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\hp\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\hp\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable

KNN Model Accuracy: 74.02%

Classification Report (KNN):
              precision    recall  f1-score   support

           0       0.81      0.84      0.83      1035
           1       0.51      0.45      0.48       374

    accuracy                           0.74      1409
   macro avg       0.66      0.65      0.65      1409
weighted avg       0.73      0.74      0.73      1409



In [23]:
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_test, y_pred))
print(classification_report(
    y_test,
    y_pred,
    target_names=["No Churn", "Churn"]
))

[[977  58]
 [179 195]]
              precision    recall  f1-score   support

    No Churn       0.85      0.94      0.89      1035
       Churn       0.77      0.52      0.62       374

    accuracy                           0.83      1409
   macro avg       0.81      0.73      0.76      1409
weighted avg       0.83      0.83      0.82      1409



In [24]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

rf_balanced = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

rf_balanced.fit(X_train, y_train)

y_pred_balanced = rf_balanced.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_balanced))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_balanced))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_balanced,
    target_names=["No Churn", "Churn"]
))


Accuracy: 0.8339247693399574

Confusion Matrix:
[[974  61]
 [173 201]]

Classification Report:
              precision    recall  f1-score   support

    No Churn       0.85      0.94      0.89      1035
       Churn       0.77      0.54      0.63       374

    accuracy                           0.83      1409
   macro avg       0.81      0.74      0.76      1409
weighted avg       0.83      0.83      0.82      1409



In [25]:
from collections import Counter

print("Before balancing:")
print(Counter(y_train))

print("Training rows:", len(X_train))

Before balancing:
Counter({0: 4139, 1: 1495})
Training rows: 5634


In [26]:

from imblearn.over_sampling import RandomOverSampler
from collections import Counter

ros = RandomOverSampler(random_state=42)

X_train_balanced, y_train_balanced = ros.fit_resample(
    X_train, y_train
)

print("After balancing:")
print(Counter(y_train_balanced))

print("Training rows before:", len(X_train))
print("Training rows after:", len(X_train_balanced))


After balancing:
Counter({0: 4139, 1: 4139})
Training rows before: 5634
Training rows after: 8278


In [27]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, f1_score, accuracy_score

model_before = RandomForestClassifier(random_state=42)
model_before.fit(X_train, y_train)

model_after = RandomForestClassifier(random_state=42)
model_after.fit(X_train_balanced, y_train_balanced)

pred_before = model_before.predict(X_test)
pred_after = model_after.predict(X_test)

print("Before Balancing")
print("Recall:", recall_score(y_test, pred_before, pos_label=1))
print("F1-score:", f1_score(y_test, pred_before, pos_label=1))
print("Accuracy:", accuracy_score(y_test, pred_before))

print("\nAfter Balancing")
print("Recall:", recall_score(y_test, pred_after, pos_label=1))
print("F1-score:", f1_score(y_test, pred_after, pos_label=1))
print("Accuracy:", accuracy_score(y_test, pred_after))


Before Balancing
Recall: 0.5213903743315508
F1-score: 0.6220095693779905
Accuracy: 0.8317955997161107

After Balancing
Recall: 0.6390374331550802
F1-score: 0.6732394366197183
Accuracy: 0.8353442157558553


In [28]:

from sklearn.metrics import confusion_matrix, classification_report

print("Before Balancing:")
print(confusion_matrix(y_test, pred_before))
print(classification_report(y_test, pred_before))

print("After Balancing:")
print(confusion_matrix(y_test, pred_after))
print(classification_report(y_test, pred_after))


Before Balancing:
[[977  58]
 [179 195]]
              precision    recall  f1-score   support

           0       0.85      0.94      0.89      1035
           1       0.77      0.52      0.62       374

    accuracy                           0.83      1409
   macro avg       0.81      0.73      0.76      1409
weighted avg       0.83      0.83      0.82      1409

After Balancing:
[[938  97]
 [135 239]]
              precision    recall  f1-score   support

           0       0.87      0.91      0.89      1035
           1       0.71      0.64      0.67       374

    accuracy                           0.84      1409
   macro avg       0.79      0.77      0.78      1409
weighted avg       0.83      0.84      0.83      1409



In [29]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

model_weighted = RandomForestClassifier(
    random_state=42,
    class_weight="balanced"
)

model_weighted.fit(X_train, y_train)

pred_weighted = model_weighted.predict(X_test)

print("Class Weight - Confusion Matrix:")
print(confusion_matrix(y_test, pred_weighted))

print("Class Weight - Classification Report:")
print(classification_report(y_test, pred_weighted))


Class Weight - Confusion Matrix:
[[974  61]
 [173 201]]
Class Weight - Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.94      0.89      1035
           1       0.77      0.54      0.63       374

    accuracy                           0.83      1409
   macro avg       0.81      0.74      0.76      1409
weighted avg       0.83      0.83      0.82      1409



In [30]:
from imblearn.over_sampling import SMOTE
from collections import Counter

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print('Class distribution before SMOTE:')
print(Counter(y_train))

print('\\nClass distribution after SMOTE:')
print(Counter(y_train_smote))

print('\\nTraining rows before:', len(X_train))
print('Training rows after SMOTE:', len(X_train_smote))

Class distribution before SMOTE:
Counter({0: 4139, 1: 1495})
\nClass distribution after SMOTE:
Counter({0: 4139, 1: 4139})
\nTraining rows before: 5634
Training rows after SMOTE: 8278


In [31]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Use the same model type and the same untouched test set for comparison.
model_smote = RandomForestClassifier(n_estimators=100, random_state=42)
model_smote.fit(X_train_smote, y_train_smote)
pred_smote = model_smote.predict(X_test)

print('SMOTE Results')
print('Accuracy:', round(accuracy_score(y_test, pred_smote), 4))
print('Precision (Churn):', round(precision_score(y_test, pred_smote, pos_label=1, zero_division=0), 4))
print('Recall (Churn):', round(recall_score(y_test, pred_smote, pos_label=1, zero_division=0), 4))
print('F1-score (Churn):', round(f1_score(y_test, pred_smote, pos_label=1, zero_division=0), 4))

SMOTE Results
Accuracy: 0.8268
Precision (Churn): 0.6776
Recall (Churn): 0.6631
F1-score (Churn): 0.6703


In [32]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

predictions = {
    'Original (Imbalanced)': pred_before,
    'Class Weight Balanced': pred_weighted,
    'RandomOverSampler': pred_after,
    'SMOTE': pred_smote
}

comparison_rows = []
for method, preds in predictions.items():
    comparison_rows.append({
        'Method': method,
        'Accuracy': accuracy_score(y_test, preds),
        'Precision (Churn)': precision_score(y_test, preds, pos_label=1, zero_division=0),
        'Recall (Churn)': recall_score(y_test, preds, pos_label=1, zero_division=0),
        'F1-score (Churn)': f1_score(y_test, preds, pos_label=1, zero_division=0)
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df[['Accuracy', 'Precision (Churn)', 'Recall (Churn)', 'F1-score (Churn)']] = comparison_df[['Accuracy', 'Precision (Churn)', 'Recall (Churn)', 'F1-score (Churn)']].round(4)
comparison_df

,Method,Accuracy,Precision (Churn),Recall (Churn),F1-score (Churn)
0,Original (Imbalanced),0.8318,0.7708,0.5214,0.6220
1,Class Weight Balanced,0.8339,0.7672,0.5374,0.6321
2,RandomOverSampler,0.8353,0.7113,0.6390,0.6732
3,SMOTE,0.8268,0.6776,0.6631,0.6703


In [33]:
from sklearn.neighbors import KNeighborsClassifier
from imblearn.over_sampling import RandomOverSampler, SMOTE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from collections import Counter
import pandas as pd

# 1) KNN trained on the original imbalanced training data
knn_original = KNeighborsClassifier(n_neighbors=5)
knn_original.fit(X_train_scaled, y_train)
pred_knn_original = knn_original.predict(X_test_scaled)

# 2) KNN + RandomOverSampler
ros_knn = RandomOverSampler(random_state=42)
X_train_ros_knn, y_train_ros_knn = ros_knn.fit_resample(X_train_scaled, y_train)

knn_ros = KNeighborsClassifier(n_neighbors=5)
knn_ros.fit(X_train_ros_knn, y_train_ros_knn)
pred_knn_ros = knn_ros.predict(X_test_scaled)

# 3) KNN + SMOTE
smote_knn = SMOTE(random_state=42)
X_train_smote_knn, y_train_smote_knn = smote_knn.fit_resample(X_train_scaled, y_train)

knn_smote = KNeighborsClassifier(n_neighbors=5)
knn_smote.fit(X_train_smote_knn, y_train_smote_knn)
pred_knn_smote = knn_smote.predict(X_test_scaled)

print('Original training distribution:', Counter(y_train))
print('After RandomOverSampler:', Counter(y_train_ros_knn))
print('After SMOTE:', Counter(y_train_smote_knn))

Original training distribution: Counter({0: 4139, 1: 1495})
After RandomOverSampler: Counter({0: 4139, 1: 4139})
After SMOTE: Counter({0: 4139, 1: 4139})


In [34]:
knn_predictions = {
    'KNN - Original Imbalanced': pred_knn_original,
    'KNN + RandomOverSampler': pred_knn_ros,
    'KNN + SMOTE': pred_knn_smote
}

knn_rows = []
for method, preds in knn_predictions.items():
    knn_rows.append({
        'Method': method,
        'Accuracy': accuracy_score(y_test, preds),
        'Precision (Churn)': precision_score(y_test, preds, pos_label=1, zero_division=0),
        'Recall (Churn)': recall_score(y_test, preds, pos_label=1, zero_division=0),
        'F1-score (Churn)': f1_score(y_test, preds, pos_label=1, zero_division=0)
    })

knn_comparison_df = pd.DataFrame(knn_rows)
metric_cols = ['Accuracy', 'Precision (Churn)', 'Recall (Churn)', 'F1-score (Churn)']
knn_comparison_df[metric_cols] = knn_comparison_df[metric_cols].round(4)
knn_comparison_df

,Method,Accuracy,Precision (Churn),Recall (Churn),F1-score (Churn)
0,KNN - Original Imbalanced,0.7402,0.5121,0.4519,0.4801
1,KNN + RandomOverSampler,0.5898,0.3473,0.6203,0.4453
2,KNN + SMOTE,0.6118,0.3646,0.6230,0.4600


In [55]:
### SVM
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

svm_model = make_pipeline(
    StandardScaler(),
    SVC(
        kernel='rbf',
        class_weight='balanced',
        random_state=42
    )
)

svm_model.fit(X_train, y_train)

y_pred_svm = svm_model.predict(X_test)

print("SVM Results")
print("Accuracy:", accuracy_score(y_test, y_pred_svm))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_svm))


SVM Results
Accuracy: 0.6600425833924769

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.68      0.74      1035
           1       0.41      0.62      0.49       374

    accuracy                           0.66      1409
   macro avg       0.62      0.65      0.62      1409
weighted avg       0.72      0.66      0.68      1409



In [57]:
## logistic regression

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# 1. Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

lr_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    )
)

lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)

print("Logistic Regression Results")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))



Logistic Regression Results
Accuracy: 0.7601135557132718

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.81      0.83      1035
           1       0.54      0.63      0.58       374

    accuracy                           0.76      1409
   macro avg       0.70      0.72      0.71      1409
weighted avg       0.77      0.76      0.77      1409



In [58]:
## Decision tree
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

dt_model = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42
)

dt_model.fit(X_train, y_train)

y_pred = dt_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))




Accuracy: 0.7473385379701917

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.70      0.80      1035
           1       0.51      0.87      0.65       374

    accuracy                           0.75      1409
   macro avg       0.73      0.79      0.72      1409
weighted avg       0.82      0.75      0.76      1409



In [61]:

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Collect predictions from all models
predictions = {
    "Random Forest": y_pred,
    "Random Forest - Class Weight": y_pred_balanced,
    "Random Forest - RandomOverSampler": pred_after,
    "Random Forest - SMOTE": pred_smote,
    "KNN": pred_knn_original,
    "KNN - RandomOverSampler": pred_knn_ros,
    "KNN - SMOTE": pred_knn_smote,
    "SVM": y_pred_svm,
    "Logistic Regression": y_pred_lr,
    "Decision Tree": y_pred
}

# Calculate evaluation metrics
results = []

for model_name, y_pred_model in predictions.items():
    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred_model),
        "Precision": precision_score(
            y_test, y_pred_model, zero_division=0
        ),
        "Recall": recall_score(
            y_test, y_pred_model, zero_division=0
        ),
        "F1-score": f1_score(
            y_test, y_pred_model, zero_division=0
        )
    })

comparison_df = pd.DataFrame(results)

# Sort by F1-score
comparison_df = comparison_df.sort_values(
    by="F1-score", ascending=False
).reset_index(drop=True)

# Display results as percentages
display(
    comparison_df.style.format({
        "Accuracy": "{:.2%}",
        "Precision": "{:.2%}",
        "Recall": "{:.2%}",
        "F1-score": "{:.2%}"
    })
)


,Model,Accuracy,Precision,Recall,F1-score
0,Random Forest - RandomOverSampler,83.53%,71.13%,63.90%,67.32%
1,Random Forest - SMOTE,82.68%,67.76%,66.31%,67.03%
2,Random Forest,74.73%,51.42%,86.90%,64.61%
3,Decision Tree,74.73%,51.42%,86.90%,64.61%
4,Random Forest - Class Weight,83.39%,76.72%,53.74%,63.21%
5,Logistic Regression,76.01%,54.13%,63.10%,58.27%
6,SVM,66.00%,40.74%,61.76%,49.10%
7,KNN,74.02%,51.21%,45.19%,48.01%
8,KNN - SMOTE,61.18%,36.46%,62.30%,46.00%
9,KNN - RandomOverSampler,58.98%,34.73%,62.03%,44.53%
